# Lab 2 — Baseline com múltiplos datasets

Neste notebook vamos rodar o mesmo pipeline de baseline para 4 datasets diferentes.

A ideia é simples: para cada dataset na lista `keys_to_run`, o código vai:
1. baixar e carregar os dados
2. limpar valores ausentes e aplicar `dropna` antes do split
3. treinar dois modelos baseline (um linear e um de floresta)
4. imprimir as métricas e salvar os resultados

No final, uma tabela compara os melhores modelos de cada dataset.

In [1]:
# 1) Bibliotecas
from __future__ import annotations

import json
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

from namex import export
import numpy as np
import pandas as pd
import pyarrow
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from lightgbm import LGBMClassifier, LGBMRegressor
#CatBoost
from catboost import CatBoostClassifier, CatBoostRegressor
import db_dtypes


In [2]:
# 2) Metadados dos datasets e funcoes  que nos ajudam a baixar, extrair e normalizar os dados

DATASETS = {
    "adult": {"task": "classification", "target": "income"},
    "bank_marketing": {"task": "classification", "target": "y"},
    "air_quality_uci": {"task": "regression", "target": "C6H6(GT)"},
    "communities_crime": {"task": "regression", "target": "ViolentCrimesPerPop"},
    "cnpq_bolsas": {"task": "classification", "target": "grande_area_conhecimento"},
}

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def download_file(url: str, output_path: Path) -> Path:
    supress_output = True

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists():
        print(f"[download] {url}")
        urlretrieve(url, output_path)
    elif not supress_output:
        print(f"[cache] {output_path}")
    return output_path


def unzip_file(zip_path: Path, extract_dir: Path) -> Path:
    supress_output = True

    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        print(f"[extract] {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text("ok\n", encoding="utf-8")
    elif not supress_output:
        print(f"[cache] {extract_dir}")
    return extract_dir


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace(STRING_MISSING_VALUES, pd.NA)
    return df



In [ ]:
def load_cnpq_bolsas(data_dir: Path, sample_size: int = 50000) -> tuple[pd.DataFrame, str, str]:
    """Carrega o dataset CNPQ com download robusto e limpeza automática."""
    import re
    import pyarrow.parquet as pq

    LINKARQUIVO = (
        "https://drive.google.com/uc?export=download"
        "&id=1kZLrjRr4z1MGx7ehWQHXUcFwgsTqVOIX"
    )

    base = data_dir / "dataset_cnpq_felps"
    base.mkdir(parents=True, exist_ok=True)

    arquivo_path = base / "br_cnpq_bolsas_microdados.parquet"

    def _download_with_gdown_or_requests(url: str, output_path: Path) -> None:
        if output_path.exists():
            return

        try:
            import gdown

            gdown.download(url, str(output_path), quiet=False, fuzzy=True)
            return
        except Exception:
            pass

        import requests

        session = requests.Session()
        match = re.search(r"(?:id=|/d/)([0-9A-Za-z_-]+)", url)
        if match:
            file_id = match.group(1)
            base_url = "https://docs.google.com/uc?export=download"
            response = session.get(base_url, params={"id": file_id}, stream=True)
            token = None
            for cookie_name, cookie_value in response.cookies.items():
                if cookie_name.startswith("download_warning"):
                    token = cookie_value
                    break
            if token:
                response = session.get(
                    base_url,
                    params={"id": file_id, "confirm": token},
                    stream=True,
                )
        else:
            response = session.get(url, stream=True)

        response.raise_for_status()
        with open(output_path, "wb") as file_handle:
            for chunk in response.iter_content(chunk_size=32768):
                if chunk:
                    file_handle.write(chunk)

    _download_with_gdown_or_requests(LINKARQUIVO, arquivo_path)

    table = pq.read_table(arquivo_path)
    if table.schema.pandas_metadata:
        table = table.replace_schema_metadata()

    df = table.to_pandas()

    colunas_remover = [
        "fonte_recurso",
        "plano_interno",
        "acao_ppa",
        "programa_ppa",
        "unidade_orcamentaria",
        "natureza_despesa",
        "categoria_nivel",
        "processo",
        "beneficiario",
        #"titulo_projeto",
        "palavra_chave",
        #"area_conhecimento",
        #"subarea_conhecimento",
        "data_inicio_processo",
        "data_fim_processo",
    ]

    target = "grande_area_conhecimento"
    task = "classification"

    df = df.drop(columns=[c for c in colunas_remover if c in df.columns], errors="ignore")

    missing_pct = df.isnull().mean()
    high_missing = missing_pct[missing_pct > 0.40].index.tolist()
    high_missing = [c for c in high_missing if c != target]
    if high_missing:
        df = df.drop(columns=high_missing, errors="ignore")

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].astype("string")

    if sample_size and len(df) > sample_size:
        df = df.sample(sample_size, random_state=42)

    if target not in df.columns:
        raise RuntimeError(f"Target '{target}' nao encontrada apos limpeza.")

    return df, target, task

In [4]:
# 3) Funcoes. para baixar os datasets automaticamente

# fiz alguns pequenos ajustes caso a caso para facilitar a leitura e evitar erros de parsing

def load_adult(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "adult"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        base / "adult.data",
    )
    test_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
        base / "adult.test",
    )

    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income",
    ]

    train_df = pd.read_csv(data_file, header=None, names=columns, skipinitialspace=True, na_values=["?", " ?"])
    test_df = pd.read_csv(test_file, header=None, names=columns, skiprows=1, skipinitialspace=True, na_values=["?", " ?"])

    df = pd.concat([train_df, test_df], ignore_index=True)
    df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
    return df, "income", "classification"


def load_bank_marketing(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "bank_marketing"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip",
        base / "bank.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    df = pd.read_csv(extract_dir / "bank-full.csv", sep=";")
    return df, "y", "classification"


def load_air_quality_uci(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "air_quality_uci"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip",
        base / "AirQualityUCI.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    candidates = list(extract_dir.rglob("AirQualityUCI.csv"))
    if not candidates:
        raise FileNotFoundError("Nao encontrei AirQualityUCI.csv dentro do zip.")

    df = pd.read_csv(candidates[0], sep=";", decimal=",")
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    for col in ["Date", "Time"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df, "C6H6(GT)", "regression"


def load_communities_crime(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "communities_crime"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data",
        base / "communities.data",
    )
    df = pd.read_csv(data_file, header=None, na_values=["?"])

    cols = [f"col_{i}" for i in range(df.shape[1])]
    cols[-1] = "ViolentCrimesPerPop"
    df.columns = cols
    # colunas  com muito problema
    df = df.drop(columns=[c for c in ["col_0", "col_1", "col_2", "col_3", "col_4"] if c in df.columns])
    return df, "ViolentCrimesPerPop", "regression"


def load_dataset(key: str, data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    if key == "adult":
        return load_adult(data_dir)
    if key == "bank_marketing":
        return load_bank_marketing(data_dir)
    if key == "air_quality_uci":
        return load_air_quality_uci(data_dir)
    if key == "communities_crime":
        return load_communities_crime(data_dir)
    if key == "cnpq_bolsas":
        return load_cnpq_bolsas(data_dir)   
    raise ValueError(f"Key desconhecida: {key}")

In [5]:
# 4) Funcoes para separar  treino e teste


def split_X_y(df: pd.DataFrame, target: str) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target]), df[target]


def build_preprocessor(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler() if scale_numeric else "passthrough", numeric_cols),
            ("cat", make_one_hot_encoder(), categorical_cols),
        ],
        remainder="drop",
    )


def make_class_weights(y: pd.Series) -> dict:
    counts = y.value_counts(dropna=False)
    total = float(counts.sum())
    n_classes = float(len(counts))
    return {label: total / (n_classes * float(count)) for label, count in counts.items() if count > 0}


def run_classification(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    class_counts = y.value_counts(dropna=False)
    stratify = y if class_counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    ) 
    catboost_class_weights = make_class_weights(y_train)
    models = {
        "logistic_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced_subsample",
            ))]),
        "lightgbm": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", LGBMClassifier(
                n_estimators=200,
                learning_rate=0.05,
                random_state=random_state,
                verbosity=-1,
                class_weight="balanced",
         ))]),
        "catboost": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", CatBoostClassifier(
                iterations=200,
                learning_rate=0.05,
                random_state=random_state,
                verbose=0,
                auto_class_weights="Balanced",  
            ))]),   


    }

    results = {}
    for name, model in models.items():
        if model is None:
            continue
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        }
        results[name] = metrics

        print(f"\n[model] {name}")
        print(metrics)
        print("Matriz de confusao:")
        print(confusion_matrix(y_test, pred))
        print(classification_report(y_test, pred))

    return results



def run_regression(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "ridge_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", Ridge(alpha=1.0)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestRegressor(n_estimators=200, random_state=random_state, n_jobs=-1)),
        ]),
        "lightgbm": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                random_state=random_state,
                verbosity=-1
        ))]), 
        "catboost": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", CatBoostRegressor(
                iterations=200,
                learning_rate=0.05,
                random_state=random_state,
                verbose=0,
            ))]),     
    }
    
    results = {}
    for name, model in models.items():
        if model is None:
            continue
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mse = mean_squared_error(y_test, pred)
        metrics = {
            "mae": float(mean_absolute_error(y_test, pred)),
            "rmse": float(np.sqrt(mse)),
            "r2": float(r2_score(y_test, pred)),
        }
        results[name] = metrics
        print(f"\n[model] {name}")
        print(metrics)

    return results



def run_one_dataset(key: str, out_dir: str = "data", test_size: float = 0.2, random_state: int = 42) -> dict:
    data_dir = Path(out_dir)
    df, target, task = load_dataset(key, data_dir=data_dir)

    print("\n" + "=" * 80)
    print(f"Dataset: {key} | Task: {task} | Target: {target}")
    print("=" * 80)
    print("Shape original:", df.shape)

    df = normalize_missing_values(df)
    rows_before = len(df)
    rows, cols = df.shape
    null_counts = df.isna().sum().sum()
    df_clean = df.dropna(axis=0, how="any").copy()
    rows_after = len(df_clean)
    print("Linhas removidas no dropna:", rows_before - rows_after)

    if rows_after == 0:
        raise RuntimeError("Dataset vazio apos dropna.")

    X, y = split_X_y(df_clean, target)

    if task == "classification":
        print("Distribuicao da target:")
        print(y.value_counts())
        results = run_classification(X, y, test_size=test_size, random_state=random_state)
    else:
        y = pd.to_numeric(y, errors="coerce")
        valid = y.notna()
        X = X.loc[valid].copy()
        y = y.loc[valid].copy()
        print("Resumo da target:\n", y.describe())
        results = run_regression(X, y, test_size=test_size, random_state=random_state)

    out_path = data_dir / f"{key}_baseline_results.json"
    out_path.write_text(
        json.dumps(
            {
                "key": key,
                "task": task,
                "target": target,
                "shape_original": list(df.shape),
                "shape_after_dropna": list(df_clean.shape),
                "test_size": test_size,
                "random_state": random_state,
                "results": results,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print("Resultados salvos em:", out_path)

    return {
        "key": key,
        "task": task,
        "target": target,
        "rows_original": int(df.shape[0]),
        "cols_original": cols,
        "rows_after_dropna": int(df_clean.shape[0]),
        "null_counts": null_counts,
        "results": results,
    }


In [6]:
# 5) Configuracao da execucao 

# Pessoal, caso fique pesado, teste em um dataet primeiro, faca sample dos dados, ou diminua o n_estimators do random forest para 50 ou 100, etc

datasets_experiment = [
    "adult",
    "bank_marketing",
    "air_quality_uci",
    "communities_crime",
    "cnpq_bolsas",
]

# Parametros globais
out_dir = "data"
test_size = 0.2
random_state = 42

datasets_experiment

['adult',
 'bank_marketing',
 'air_quality_uci',
 'communities_crime',
 'cnpq_bolsas']

In [7]:
# 6) Loop principal

all_runs = []

for key in datasets_experiment:
    try:
        run_info = run_one_dataset(
            key=key,
            out_dir=out_dir,
            test_size=test_size,
            random_state=random_state,
        )
        all_runs.append(run_info)
    except Exception as e:
        print(f"\n[ERRO] key={key}: {e}")
        all_runs.append({"key": key, "task": DATASETS[key]["task"], "error": str(e)})


# --- Monta tabelas separadas por tipo de tarefa ---

classification_rows = []
regression_rows = []

for r in all_runs:
    base = {"key": r["key"], "best_model": "", "error": r.get("error", "")}

    if "error" in r and "results" not in r:
        # Dataset falhou: adiciona na tabela correta com metricas vazias
        if r.get("task") == "classification":
            classification_rows.append({**base, "accuracy": None, "macro_f1": None, "weighted_f1": None})
        else:
            regression_rows.append({**base, "mae": None, "rmse": None, "r2": None})
        continue

    if r["task"] == "classification":
        best = max(r["results"], key=lambda m: r["results"][m]["weighted_f1"])
        m = r["results"][best]
        classification_rows.append({
            "key": r["key"],
            "best_model": best,
            "accuracy": round(m["accuracy"], 4),
            "macro_f1": round(m["macro_f1"], 4),
            "weighted_f1": round(m["weighted_f1"], 4),
            "error": "",
        })
    else:
        best = max(r["results"], key=lambda m: r["results"][m]["r2"])
        m = r["results"][best]
        regression_rows.append({
            "key": r["key"],
            "best_model": best,
            "mae": round(m["mae"], 4),
            "rmse": round(m["rmse"], 4),
            "r2": round(m["r2"], 4),
            "error": "",
        })

# Exibe  resultados e salva CSVs 

data_path = Path(out_dir)

if classification_rows:
    clf_df = pd.DataFrame(classification_rows, columns=["key", "best_model", "accuracy", "macro_f1", "weighted_f1", "error"])
    print("\n=== Classificacao ===")
    display(clf_df)
    clf_path = data_path / "classification_summary.csv"
    clf_df.to_csv(clf_path, index=False)
    print("Salvo em:", clf_path)

if regression_rows:
    reg_df = pd.DataFrame(regression_rows, columns=["key", "best_model", "mae", "rmse", "r2", "error"])
    print("\n=== Regressao ===")
    display(reg_df)
    reg_path = data_path / "regression_summary.csv"
    reg_df.to_csv(reg_path, index=False)
    print("Salvo em:", reg_path)


Dataset: adult | Task: classification | Target: income
Shape original: (48842, 15)
Linhas removidas no dropna: 3620
Distribuicao da target:
income
<=50K    34014
>50K     11208
Name: count, dtype: Int64

[model] logistic_regression
{'accuracy': 0.8091763405196241, 'macro_f1': 0.7742666199497235, 'weighted_f1': 0.8190299698286431}
Matriz de confusao:
[[5438 1365]
 [ 361 1881]]
              precision    recall  f1-score   support

       <=50K       0.94      0.80      0.86      6803
        >50K       0.58      0.84      0.69      2242

    accuracy                           0.81      9045
   macro avg       0.76      0.82      0.77      9045
weighted avg       0.85      0.81      0.82      9045


[model] random_forest
{'accuracy': 0.8527363184079602, 'macro_f1': 0.7892272326489482, 'weighted_f1': 0.8475686103220306}
Matriz de confusao:
[[6339  464]
 [ 868 1374]]
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.90      6803
        >50K  

c:\Users\felip\miniconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[model] lightgbm
{'accuracy': 0.8313985627418463, 'macro_f1': 0.798214325238461, 'weighted_f1': 0.839477495154748}
Matriz de confusao:
[[5594 1209]
 [ 316 1926]]
              precision    recall  f1-score   support

       <=50K       0.95      0.82      0.88      6803
        >50K       0.61      0.86      0.72      2242

    accuracy                           0.83      9045
   macro avg       0.78      0.84      0.80      9045
weighted avg       0.86      0.83      0.84      9045


[model] catboost
{'accuracy': 0.8221116639027087, 'macro_f1': 0.7891114701865924, 'weighted_f1': 0.8311779552450625}
Matriz de confusao:
[[5507 1296]
 [ 313 1929]]
              precision    recall  f1-score   support

       <=50K       0.95      0.81      0.87      6803
        >50K       0.60      0.86      0.71      2242

    accuracy                           0.82      9045
   macro avg       0.77      0.83      0.79      9045
weighted avg       0.86      0.82      0.83      9045

Resultados salvos 

c:\Users\felip\miniconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[model] lightgbm
{'accuracy': 0.8330146590184832, 'macro_f1': 0.7895568324227804, 'weighted_f1': 0.8416696539365612}
Matriz de confusao:
[[1010  202]
 [  60  297]]
              precision    recall  f1-score   support

          no       0.94      0.83      0.89      1212
         yes       0.60      0.83      0.69       357

    accuracy                           0.83      1569
   macro avg       0.77      0.83      0.79      1569
weighted avg       0.86      0.83      0.84      1569


[model] catboost
{'accuracy': 0.8330146590184832, 'macro_f1': 0.7948131007191659, 'weighted_f1': 0.8430587910380969}
Matriz de confusao:
[[992 220]
 [ 42 315]]
              precision    recall  f1-score   support

          no       0.96      0.82      0.88      1212
         yes       0.59      0.88      0.71       357

    accuracy                           0.83      1569
   macro avg       0.77      0.85      0.79      1569
weighted avg       0.88      0.83      0.84      1569

Resultados salvos em

...


[model] lightgbm
{'accuracy': 0.9996580417189103, 'macro_f1': 0.9986350386107123, 'weighted_f1': 0.9996580058613401}
Matriz de confusao:
[[1208    0    0    0    0    0    0    0    0    0]
 [   0 1173    0    0    0    0    0    0    0    0]
 [   0    0  982    0    0    0    0    0    0    0]
 [   0    0    0 1921    0    0    0    0    0    0]
 [   0    0    0    0 1015    0    0    0    0    0]
 [   0    0    0    0    0  586    0    0    1    0]
 [   0    0    0    0    0    0 1174    0    0    0]
 [   0    0    0    0    0    0    0  292    0    0]
 [   0    0    0    0    0    0    0    0  289    0]
 [   0    0    0    0    0    0    0    0    2  130]]
                             precision    recall  f1-score   support

          CIÊNCIAS AGRÁRIAS       1.00      1.00      1.00      1208
        CIÊNCIAS BIOLÓGICAS       1.00      1.00      1.00      1173
          CIÊNCIAS DA SAÚDE       1.00      1.00      1.00       982
 CIÊNCIAS EXATAS E DA TERRA       1.00      1.00      

,key,best_model,accuracy,macro_f1,weighted_f1,error
0,adult,random_forest,0.8527,0.7892,0.8476,
1,bank_marketing,catboost,0.8330,0.7948,0.8431,
2,cnpq_bolsas,logistic_regression,0.9999,0.9996,0.9999,


Salvo em: data\classification_summary.csv

=== Regressao ===


,key,best_model,mae,rmse,r2,error
0,air_quality_uci,random_forest,0.0694,0.2302,0.9991,
1,communities_crime,random_forest,0.1110,0.1458,0.7041,


Salvo em: data\regression_summary.csv


# Criação de tabela resumo

In [8]:
df_tabela_resumo = pd.DataFrame(all_runs)
df_tabela_resumo.to_csv(data_path / "tabela_resumo_completa.csv", index=False)

# 1. Preparação dos Dados

In [9]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from jenga.corruptions.generic import MissingValues
from gain import gain
from typing import Literal

c:\Users\felip\miniconda3\envs\tf_env\lib\site-packages\jenga\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound



Instructions for updating:
non-resource variables are not supported in the long term


"Tentamos inicialmente utilizar o Datawig, porém o método apresentou incompatibilidade de dependências com as versões atuais das bibliotecas exigidas pelo protocolo (scikit-learn >= 1.0). Para garantir a integridade do experimento e a utilização dos imputadores multivariados do scikit-learn, optamos por substituir o Datawig pelo método [GAIN ou DiffPuter]".

In [10]:
strategyType = Literal["knn3","knn5","knn7", "iterative", "gain", "mean", "median", "most_frequent", "constant"]
simple_imputer_strategies = ["mean", "median", "most_frequent", "constant"]
percents = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
mecanism = ['MCAR', 'MAR', 'MNAR']
resultados = []

def impute_numeric_data(df: pd.DataFrame, num_cols: list, strategy: str, random_state: int) -> pd.DataFrame:

    if( strategy in simple_imputer_strategies):
        imputer = SimpleImputer(strategy=strategy)
        df_imputed = pd.DataFrame(imputer.fit_transform(df[num_cols]),index=df.index, columns=num_cols)

    if( strategy in ["knn3", "knn5", "knn7"]):
        n_neighbors = int(strategy[-1])
        imputer = KNNImputer(n_neighbors=n_neighbors)
        df_imputed = pd.DataFrame(imputer.fit_transform(df[num_cols]),index=df.index, columns=num_cols)

    if( strategy == "iterative"):
        # TODO: justifique as configurações
        # Max_iter: 10 para 30 por causa de warnings
        imputer = IterativeImputer(max_iter=30,random_state=random_state)
        df_imputed = pd.DataFrame(imputer.fit_transform(df[num_cols]),index=df.index, columns=num_cols)

    if( strategy == "gain"):
        gain_parameters = {
            'batch_size': 128,
            'hint_rate': 0.9,
            'alpha': 100,
            'iterations': 5000 
        }
        data_numpy = df[num_cols].to_numpy(dtype=float)
        imputed_data_numpy = gain(data_numpy, gain_parameters)
    
        df_imputed = pd.DataFrame(
            imputed_data_numpy, 
            columns=num_cols, 
            index=df.index
        )

    return df_imputed

def impute_data(df: pd.DataFrame, strategy: str, random_state: int) -> pd.DataFrame:
    
    num_cols = df.select_dtypes(include=['number']).columns
    cat_cols = df.select_dtypes(exclude=['number']).columns

    df_imputed_cat = pd.DataFrame(index=df.index)
    df_imputed_num = pd.DataFrame(index=df.index)

    if(len(cat_cols) > 0): # TEM MAIS ESTRATÉGIAS ???
        cat_imputer = SimpleImputer(strategy='most_frequent') # para textos
        df_imputed_cat = pd.DataFrame(cat_imputer.fit_transform(df[cat_cols]), index=df.index, columns=cat_cols)

    if(len(num_cols) > 0):
        df_imputed_num = impute_numeric_data(df, num_cols, strategy, random_state)

    df_final = df_imputed_num.join(df_imputed_cat)
    return df_final[df.columns] # garante a mesma ordem de colunas do df original

def main_experiment(
        key: str,
        strategy: strategyType,
        mecanism: str,
        percent: float,
        out_dir: str = "data",
        test_size: float = 0.2,
        random_state: int = 42,
    ) -> dict:
    data_dir = Path(out_dir)
    df, target, task = load_dataset(key, data_dir=data_dir)
    df_baseline = df.dropna(axis=1, how="any").dropna(axis=0, how="all")

    X_baseline = df_baseline.drop(columns=[target])
    Y = df_baseline[target]

    X_injected = jenga_inject(X_baseline, mecanism=mecanism, percent=percent)

    X_imputed = impute_data(X_injected, strategy=strategy, random_state=random_state)
    print("         Total de missing values após imputação: " + str(X_imputed.isnull().sum().sum())) 

#PROMPT DE IA    
def jenga_inject(X_baseline: pd.DataFrame, mecanism: str, percent: float) -> pd.DataFrame:

    # Cria uma cópia para não destruir o dado original (ground truth)
    X_injetado = X_baseline.copy()
    
    # O Jenga aplica a regra coluna por coluna
    for coluna in X_injetado.columns:
        
        # Instancia o corruptor do Jenga para a coluna atual
        # Ele aceita exatamente os termos 'MCAR', 'MAR' e 'MNAR'
        corruptor = MissingValues(column=coluna, fraction=percent, missingness=mecanism)
        
        # Aplica a injeção na coluna e atualiza o DataFrame
        X_injetado = corruptor.transform(X_injetado)
        
    print("         Total de missing values injetados: " + str(X_injetado.isnull().sum().sum()))    # Apenas para verificar quantos valores faltantes foram injetados na coluna
    return X_injetado

def quality_assessment(X_baseline: pd.DataFrame, X_injected: pd.DataFrame, X_imputed: pd.DataFrame) -> dict:
    #TODO: Avaliar Qualidade
    pass

for mec in mecanism:
    print(f"\n\n=== MECANISMO: {mec} ===")
    for perc in percents:
        print(f"  Percentual de missing: {perc:.0%}")
        for key in datasets_experiment:
            print(f"      Dataset: {key}")
            main_experiment(
                key,
                strategy="constant",
                out_dir="data",
                test_size=0.2,
                random_state=42,
                mecanism=mec,
                percent=perc,
            )
        
        



=== MECANISMO: MCAR ===
  Percentual de missing: 1%
      Dataset: adult
         Total de missing values injetados: 5368
         Total de missing values após imputação: 0
      Dataset: bank_marketing
         Total de missing values injetados: 7232
         Total de missing values após imputação: 0
      Dataset: air_quality_uci
         Total de missing values injetados: 1116
         Total de missing values após imputação: 0
      Dataset: communities_crime
         Total de missing values injetados: 1881
         Total de missing values após imputação: 0
      Dataset: cnpq_bolsas
         Total de missing values injetados: 2500
         Total de missing values após imputação: 0
  Percentual de missing: 5%
      Dataset: adult
         Total de missing values injetados: 26862
         Total de missing values após imputação: 0
      Dataset: bank_marketing
         Total de missing values injetados: 36160
         Total de missing values após imputação: 0
      Dataset: air_qual